# Day 4 — Data Quality & Cleaning

**Goal:** Validate the UCI Credit Card Default dataset is production-ready and document every cleaning decision for model governance.

## Sections
1. Load Data
2. Missing Value Strategy
3. Invalid Value Detection
4. Outlier Detection (IQR)
5. Data Leakage Audit
6. Summary

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_ingestion import load_data

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

In [ ]:
df = load_data(
    raw_path=PROJECT_ROOT / 'data/raw/UCI_Credit_Card.xls',
    processed_path=PROJECT_ROOT / 'data/processed/credit_default_cleaned.parquet'
)
print(f'Shape: {df.shape}')
df.head(3)

## 1. Missing Value Strategy

> **Interview tip:** Always state your imputation rationale. Median for skewed continuous features, mode for categorical. Document the decision — SR 11-7 requires it.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
summary = summary[summary.missing_count > 0]

if summary.empty:
    print('No missing values — dataset is complete.')
else:
    print(summary)
    # Imputation strategy (applied only if missing values exist)
    for col in summary.index:
        if summary.loc[col, 'missing_pct'] > 50:
            print(f'{col}: >50% missing → drop column')
            df = df.drop(columns=[col])
        elif df[col].dtype in ['float64', 'int64']:
            strategy = 'median' if df[col].skew() > 1 else 'mean'
            fill_val = df[col].median() if strategy == 'median' else df[col].mean()
            df[col] = df[col].fillna(fill_val)
            print(f'{col}: filled with {strategy} ({fill_val:.2f})')
        else:
            fill_val = df[col].mode()[0]
            df[col] = df[col].fillna(fill_val)
            print(f'{col}: filled with mode ({fill_val})')

## 2. Invalid Value Detection

Checks: AGE out of range, LIMIT_BAL non-positive, EDUCATION / MARRIAGE undocumented codes.

In [ ]:
issues = {}

# AGE: must be 18–100
invalid_age = df[(df['AGE'] < 18) | (df['AGE'] > 100)]
issues['AGE out of range (<18 or >100)'] = len(invalid_age)

# LIMIT_BAL: must be positive
invalid_limit = df[df['LIMIT_BAL'] <= 0]
issues['LIMIT_BAL <= 0'] = len(invalid_limit)

# EDUCATION: valid values are 1, 2, 3, 4 (after binning in data_ingestion.py)
valid_edu = {1, 2, 3, 4}
invalid_edu = df[~df['EDUCATION'].isin(valid_edu)]
issues['EDUCATION invalid codes'] = len(invalid_edu)

# MARRIAGE: valid values are 1, 2, 3 (after binning in data_ingestion.py)
valid_mar = {1, 2, 3}
invalid_mar = df[~df['MARRIAGE'].isin(valid_mar)]
issues['MARRIAGE invalid codes'] = len(invalid_mar)

print('Invalid value audit:')
for check, count in issues.items():
    status = 'PASS' if count == 0 else f'FAIL — {count} rows'
    print(f'  {check}: {status}')

# Fix AGE and LIMIT_BAL if violations exist
if len(invalid_age) > 0:
    df = df[(df['AGE'] >= 18) & (df['AGE'] <= 100)]
    print(f'\nDropped {len(invalid_age)} rows with invalid AGE.')

if len(invalid_limit) > 0:
    df = df[df['LIMIT_BAL'] > 0]
    print(f'Dropped {len(invalid_limit)} rows with non-positive LIMIT_BAL.')

In [ ]:
# AGE range summary
print(f"AGE range: {df['AGE'].min()} – {df['AGE'].max()}")
print(f"LIMIT_BAL range: {df['LIMIT_BAL'].min():,.0f} – {df['LIMIT_BAL'].max():,.0f}")
print(f"EDUCATION values: {sorted(df['EDUCATION'].unique())}")
print(f"MARRIAGE values:  {sorted(df['MARRIAGE'].unique())}")

## 3. Outlier Detection (IQR Method)

We flag extreme BILL_AMT values using the IQR rule. For credit data, high bill amounts are **natural outliers** (large credit limits exist legitimately) — we flag but do not remove.

> **Interview tip:** "I used IQR to flag rather than remove, because extreme bill amounts reflect high-limit cardholders, not data errors. Removing them would introduce selection bias toward lower-income segments."

In [ ]:
bill_cols = ['BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6']

outlier_summary = []

for col in bill_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = n_outliers / len(df) * 100
    outlier_summary.append({
        'column': col, 'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'lower_fence': lower, 'upper_fence': upper,
        'n_outliers': n_outliers, 'pct_outliers': round(pct, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).set_index('column')
print('IQR Outlier Summary — BILL_AMT columns:')
print(outlier_df[['n_outliers', 'pct_outliers', 'lower_fence', 'upper_fence']])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(bill_cols):
    axes[i].boxplot(df[col].dropna(), vert=True)
    axes[i].set_title(f'{col}')
    axes[i].set_ylabel('Amount (NTD)')
    n = outlier_df.loc[col, 'n_outliers']
    pct = outlier_df.loc[col, 'pct_outliers']
    axes[i].set_xlabel(f'{n} outliers ({pct}%) — kept (natural)')

plt.suptitle('BILL_AMT Outlier Detection (IQR Method)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/07_bill_amt_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Leakage Audit

**Target:** `default` = whether the client defaulted in the month AFTER the observation period.

A feature causes **leakage** if it encodes information only available after the default event occurs.

| Feature Group | Time Period | Leakage Risk |
|---------------|-------------|-------------|
| `LIMIT_BAL` | Set before observation period | None |
| `SEX`, `EDUCATION`, `MARRIAGE`, `AGE` | Static demographics | None |
| `PAY_0` to `PAY_6` | Apr–Sep 2005 payment status | None — target is Oct 2005 |
| `BILL_AMT1` to `BILL_AMT6` | Apr–Sep 2005 bill amounts | None |
| `PAY_AMT1` to `PAY_AMT6` | Apr–Sep 2005 payments made | None |
| `default` | Oct 2005 | **This is the target** |

**Verdict: No leakage.** All features reflect the state of the account *before* the target period.

In [ ]:
# Verify no feature is a direct proxy for the target
feature_cols = [c for c in df.columns if c != 'default']
corr_with_target = df[feature_cols].corrwith(df['default']).abs().sort_values(ascending=False)

print('Top 10 features by |correlation| with target:')
print(corr_with_target.head(10).to_string())
print('\nNo correlation is 1.0 — no leaked target proxy.')

## 5. Summary

In [ ]:
print('=== DATA QUALITY SUMMARY ===')
print(f'Final shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Rows with valid AGE (18–100): {((df["AGE"] >= 18) & (df["AGE"] <= 100)).sum()}')
print(f'Rows with valid LIMIT_BAL (>0): {(df["LIMIT_BAL"] > 0).sum()}')
print(f'EDUCATION valid codes only: {df["EDUCATION"].isin([1,2,3,4]).all()}')
print(f'MARRIAGE valid codes only: {df["MARRIAGE"].isin([1,2,3]).all()}')
print(f'Data leakage: None')
print(f'Outlier action: Flagged (natural) — no rows removed')

In [ ]:
# Save validated dataset
out_path = PROJECT_ROOT / 'data/processed/credit_default_validated.parquet'
df.to_parquet(out_path, index=True)
print(f'Validated dataset saved to {out_path}')